
# XGBoost Comparison with the Same Features: `Cabin_mapped` vs `Cabin_reduced` (+ `sex`)

This notebook reproduces the setup from your original workflow and compares **two XGBoost models** trained on the **same feature set theme**:

1. **High Cardinality** — Features: `Cabin_mapped` (numeric ID for each unique cabin) + `sex`  
2. **Reduced Cardinality** — Features: `Cabin_reduced` (coarse cabin grouping) + `sex`

We report:

- Unique category counts for `Cabin_mapped` and `Cabin_reduced`
- Encoded feature dimensionality after preprocessing
- Train/test performance (Accuracy, ROC AUC)
- Fit time

> Dataset expected at: `/mnt/data/titanic.csv`


In [ ]:

import os, time, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, roc_auc_score

from xgboost import XGBClassifier

DATA_PATH = '/mnt/data/titanic.csv'
assert os.path.exists(DATA_PATH), f"File not found: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)
df.head()


## Basic cleaning and feature engineering

In [ ]:

# Ensure target exists and drop rows with missing target
assert 'Survived' in df.columns, 'Expected target column Survived not found.'
df = df.dropna(subset=['Survived']).copy()

# Normalize sex to lowercase string category
if 'sex' in df.columns:
    df['sex'] = df['sex'].astype(str).str.lower()
elif 'Sex' in df.columns:
    df['sex'] = df['Sex'].astype(str).str.lower()
else:
    raise ValueError('sex/Sex column not found.')

# Cabin engineering -----------------------------------------------------------
# High-cardinality mapping: each unique cabin string -> unique integer ID
def build_cabin_mapping(series: pd.Series):
    uniq = series.astype(str).unique()
    # retain NaN handling separately; we won't map NaN here
    mapping = {k: i for i, k in enumerate([u for u in uniq if u != 'nan'], start=0)}
    return mapping

# Reduced-cardinality: take deck letter (first alpha char) else 'U' (unknown)
def cabin_to_deck(series: pd.Series):
    def deck(s):
        if pd.isna(s): 
            return 'U'  # unknown
        s = str(s)
        # find first alphabetic character as deck
        for ch in s:
            if ch.isalpha():
                return ch.upper()
        return 'U'
    return series.apply(deck)

# Build features on the FULL df first (we'll split later to avoid leakage in mapping where needed)
df['Cabin'] = df['Cabin'] if 'Cabin' in df.columns else df.get('cabin', np.nan)
# For mapping, we will create after split to avoid leakage.
df['Cabin_reduced'] = cabin_to_deck(df['Cabin'])

# Keep only columns we need
use_cols = ['Survived', 'sex', 'Cabin', 'Cabin_reduced']
df_small = df[use_cols].copy()

df_small.head()


## Train / Test split (stratified on target)

In [ ]:

X = df_small.drop(columns=['Survived'])
y = df_small['Survived'].astype(int).values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build mapping for Cabin_mapped **from training data only** to avoid leakage
cabin_train = X_train['Cabin']
cabin_mapping = {k: i for i, k in enumerate(sorted(cabin_train.dropna().astype(str).unique()))}

# Create Cabin_mapped using training mapping; unseen in test -> -1
X_train = X_train.copy()
X_test = X_test.copy()

X_train['Cabin_mapped'] = X_train['Cabin'].astype(str).map(cabin_mapping)
X_test['Cabin_mapped']  = X_test['Cabin'].astype(str).map(cabin_mapping)

# Unseen cabin values in test get NaN -> fill with -1 later in numeric imputer
X_train[['sex','Cabin','Cabin_reduced','Cabin_mapped']].head()


## Cardinality snapshot

In [ ]:

card_high = X_train['Cabin'].dropna().astype(str).nunique()
card_low  = X_train['Cabin_reduced'].dropna().astype(str).nunique()

print('Unique cabins (high-card mapping source):', card_high)
print('Unique reduced cabins (deck letters):', card_low)

# Show top few values
print('\nSample mapping entries (Cabin -> id):', list(cabin_mapping.items())[:10])
X_train['Cabin_reduced'].value_counts().head(10)


## Pipelines

In [ ]:

def pipeline_high():
    # Features: Cabin_mapped (numeric), sex (categorical)
    num_features = ['Cabin_mapped']
    cat_features = ['sex']

    pre = ColumnTransformer(
        transformers=[
            ('num', SimpleImputer(strategy='most_frequent'), num_features),
            ('cat', Pipeline([
                ('impute', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore', sparse=True))
            ]), cat_features)
        ]
    )

    clf = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        tree_method='hist'
    )
    return Pipeline([('pre', pre), ('clf', clf)]), pre, num_features, cat_features


def pipeline_low():
    # Features: Cabin_reduced (categorical), sex (categorical)
    num_features = []  # no numeric cabin in this variant
    cat_features = ['Cabin_reduced', 'sex']

    pre = ColumnTransformer(
        transformers=[
            ('num', 'drop', num_features),
            ('cat', Pipeline([
                ('impute', SimpleImputer(strategy='most_frequent')),
                ('ohe', OneHotEncoder(handle_unknown='ignore', sparse=True))
            ]), cat_features)
        ]
    )

    clf = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        tree_method='hist'
    )
    return Pipeline([('pre', pre), ('clf', clf)]), pre, num_features, cat_features


## Train & Evaluate

In [ ]:

def fit_eval(pipe, Xtr, ytr, Xte, yte, label=''):
    t0 = time.time()
    pipe.fit(Xtr, ytr)
    t_fit = time.time() - t0

    y_pred = pipe.predict(Xte)
    y_proba = pipe.predict_proba(Xte)[:,1]

    acc = accuracy_score(yte, y_pred)
    try:
        auc = roc_auc_score(yte, y_proba)
    except ValueError:
        auc = np.nan

    # Get OHE dimension for categorical part
    pre = pipe.named_steps['pre']
    ohe = None
    for name, trans, cols in pre.transformers_:
        if name == 'cat':
            ohe = trans.named_steps.get('ohe', None)
    encoded_dim = int(sum(len(c) for c in ohe.categories_)) if ohe is not None else 0

    return {
        'label': label,
        'fit_time_sec': t_fit,
        'accuracy': acc,
        'roc_auc': auc,
        'encoded_features_after_ohe': encoded_dim
    }, pipe

# Build datasets for each variant
Xtr_high = X_train[['Cabin_mapped','sex']].copy()
Xte_high = X_test[['Cabin_mapped','sex']].copy()

Xtr_low = X_train[['Cabin_reduced','sex']].copy()
Xte_low = X_test[['Cabin_reduced','sex']].copy()

pipe_h, pre_h, num_h, cat_h = pipeline_high()
metrics_high, fitted_h = fit_eval(pipe_h, Xtr_high, y_train, Xte_high, y_test, label='High-card: Cabin_mapped + sex')
metrics_high


In [ ]:

pipe_l, pre_l, num_l, cat_l = pipeline_low()
metrics_low, fitted_l = fit_eval(pipe_l, Xtr_low, y_train, Xte_low, y_test, label='Reduced: Cabin_reduced + sex')
metrics_low


## Compare Results

In [ ]:

summary = pd.DataFrame([metrics_high, metrics_low])
display(summary)

def bar(metric):
    plt.figure()
    plt.bar(summary['label'], summary[metric])
    plt.title(metric)
    plt.ylabel(metric)
    plt.xticks(rotation=10, ha='right')
    plt.tight_layout()
    plt.show()

for m in ['encoded_features_after_ohe','fit_time_sec','accuracy','roc_auc']:
    bar(m)



## Notes

- **High cardinality (`Cabin_mapped`)**: Each unique cabin gets its own numeric ID. This keeps the pipeline compact (one numeric feature + `sex`) but may let the model overfit to rare cabins if they correlate spuriously with the target.
- **Reduced cardinality (`Cabin_reduced`)**: Using deck letter (A–G, etc.) or `'U'` for unknown collapses many distinct cabins into a few groups. This typically reduces noise and improves generalization.
- The encoded dimension you see is the one-hot size for **categorical** features (here, `sex` in the high-card setup and `sex` + `Cabin_reduced` in the reduced setup).
- To go further:
  - Try **target encoding** on `Cabin_reduced` (instead of one-hot) and compare.
  - Tighten XGBoost regularization (e.g., increase `min_child_weight`, add `reg_alpha`) to damp splits on small groups.
  - Ensure robust handling of **unseen cabins** at inference: map to `-1` or route to `'U'` / `'Other'` bucket before encoding.
